# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import logging
import requests
from agents.deals import Deal, DealSelection, Opportunity, ScrapedDeal
load_dotenv(override=True)
openAI = OpenAI()
Model = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:04<00:00,  1.54s/it]


In [3]:
len(deals)

20

In [5]:
deals[10].describe()

'Title: Pokémon Winds and Waves details: Region, starters, Mr. Windychu, Ms. Wavychu, more\nDetails: The brand new mainline Gen Pokémon Nintendo Switch 2 games have been officially revealed. As one would expect and just about everyone hoped for, the new Pokémon Winds and Pokémon Waves headlined this morning’s Pokémon Presents showcase and really finished the show off with a bang. We have already featured the reveal of the two games, but a series of additional details have emerged since then, so let’s dive in! more…\nFeatures: \nURL: https://9to5toys.com/2026/02/27/pokemon-winds-waves-details/'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: RayNeo Air 4 Pro AR/XR 3D Movies & Gaming Smart Glasses $249 & More + Free Shipping
Details: Looks like a pretty noticeable improvement over the previous 3s Pro model, and is currently $50 off when you clip the coupon. There's also the Justice/Batman edition that gives you the Batman mask...
Features: 
URL: https://slickdeals.net/f/19255051-tcl-rayneo-air-4-pro-ar-xr-glasses-20

In [9]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [10]:
response = openAI.chat.completions.parse(model=Model, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description="RayNeo Air 4 Pro AR/XR smart glasses designed for immersive 3D movies and gaming with a 201° HDR10 video display driven by the Vision 4000 chip. The headset includes integrated audio tuned by Bang & Olufsen, supports mixed reality experiences, and offers special edition cosmetic variants such as themed Justice/Batman designs. It's presented as a compact wearable for extended media and gaming sessions with improved optics over prior models.", price=249.0, url='https://slickdeals.net/f/19255051-tcl-rayneo-air-4-pro-ar-xr-glasses-201-hdr10-video-display-vision-4000-chip-audio-by-bang-amp-olufsen-3d-movies-amp-gaming-smart-glasses?utm_source=rss&utm_content=fp&utm_medium=RSS2'), Deal(product_description='Luigi’s Mansion 2 HD for Nintendo Switch available in physical or digital format. This remaster brings updated high-definition visuals and quality-of-life improvements to the GameCube classic, following Luigi as he explores haunted mansions, s

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

RayNeo Air 4 Pro AR/XR smart glasses designed for immersive 3D movies and gaming with a 201° HDR10 video display driven by the Vision 4000 chip. The headset includes integrated audio tuned by Bang & Olufsen, supports mixed reality experiences, and offers special edition cosmetic variants such as themed Justice/Batman designs. It's presented as a compact wearable for extended media and gaming sessions with improved optics over prior models.
249.0
https://slickdeals.net/f/19255051-tcl-rayneo-air-4-pro-ar-xr-glasses-201-hdr10-video-display-vision-4000-chip-audio-by-bang-amp-olufsen-3d-movies-amp-gaming-smart-glasses?utm_source=rss&utm_content=fp&utm_medium=RSS2

Luigi’s Mansion 2 HD for Nintendo Switch available in physical or digital format. This remaster brings updated high-definition visuals and quality-of-life improvements to the GameCube classic, following Luigi as he explores haunted mansions, solves puzzles, and uses his Poltergust to capture ghosts. The package provides the full s

In [2]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 20 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [15]:
result

DealSelection(deals=[Deal(product_description='RayNeo Air 4 Pro are AR/XR smart glasses designed for immersive 3D movies and gaming. They feature a 201 HDR10 video display driven by the Vision 4000 chip and integrate audio tuned by Bang & Olufsen. The glasses emphasize a wearable theater experience with improved optics over prior models and special themed editions available.', price=249.0, url='https://slickdeals.net/f/19255051-tcl-rayneo-air-4-pro-ar-xr-glasses-201-hdr10-video-display-vision-4000-chip-audio-by-bang-amp-olufsen-3d-movies-amp-gaming-smart-glasses?utm_source=rss&utm_content=fp&utm_medium=RSS2'), Deal(product_description='Luigi’s Mansion 2 HD for Nintendo Switch is a remastered action-adventure game that follows Luigi as he explores haunted mansions using the Poltergust vacuum to capture ghosts and solve puzzles. The HD release updates visuals and performance for the Switch platform and is available as either a physical cartridge or digital download for the console.', pri

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [3]:
load_dotenv(override=True)

True

In [17]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [18]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [19]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [20]:
push("hi")

Push: hi


In [3]:
from agents.messaging_agent import MessagingAgent
agent = MessagingAgent()

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and OpenAI


In [3]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and OpenAI
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [4]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
